# WHILE コンパイラとスタック式アバカスマシン — `UNIVS in UNIVS`

この notebook では、次の実験を行います。

1. WHILE 言語の小さな抽象構文（AST）を用意する。
2. その WHILE プログラムを **CALL/RET 付きスタック式アバカスマシン**へコンパイルする `compile_program` を作る。
3. `listcode`, `listentry`, `listreplace` などを含む `UNIVS` 自身を WHILE プログラムとして記述する。
4. それをスタック式アバカスマシンへコンパイルし、命令数とコードの大きさを測る。
5. コンパイルされた `UNIVS` をスタック式アバカスマシン上で動かし、その `UNIVS` にさらに小さなスタック式アバカスマシンのプログラムを解釈させる。

つまり、構図としては

```text
WHILE で書かれた UNIVS
        ↓ compile_program
スタック式アバカスマシン版 UNIVS
        ↓ 実行
内側のスタック式アバカスマシンプログラムを解釈
```

です。

昔の **Lisp in Lisp** に似ていますが、こちらは「手続型言語のプログラムを、その言語用のコンパイラで機械語へ落とし、その機械語上で万能プログラムを動かす」という版です。

> 注：この notebook の `compile_program` の実装自体は、まず実験を成立させるため Python で書いてあります。ただし扱っているデータは自然数・有限列・構文木で、使っている処理は WHILE で実装可能です。最後の節で、これを WHILE 自身へ移す self-hosting の段階を整理します。


## 1. コンパイル先：スタック式アバカスマシン

命令は次の 5 種類です。

```text
[1,n,next]        n+
[2,n,next,enext]  n-
[0]               END
[3,a,next,m]      CALL
[4]               RET
```

各関数の `next`, `enext` は **関数内相対番地**です。関数 `a` の絶対開始番地は `funs[a]` に入っています。

CALL では `(fun,next,fp)` を制御スタックに積み、`fp` をずらして新しいスタックフレームに移ります。RET ではこの 3 つを復元します。


## 2. WHILE の小さな AST と、スタック式アバカスマシンへのコンパイラ

In [1]:
from dataclasses import dataclass

# AST constructors
def V(x): return ('var', x)
def C(n): return ('const', n)
def Succ(e): return ('succ', e)
def Pred(e): return ('pred', e)
def Call(f,*args): return ('call', f, list(args))
def Assign(x,e): return ('assign', x, e)
def If(cond, th, el=None): return ('if', cond, th, [] if el is None else el)
def While(cond, body): return ('while', cond, body)
def F(name,args,body,ret): return {'name':name,'args':args,'body':body,'ret':ret}

class Asm:
    def __init__(self):
        self.items=[]; self.c=0
    def fresh(self,p='L'):
        self.c+=1; return f'__{p}_{self.c}'
    def label(self,l): self.items.append(('label',l))
    def emit(self,inst): self.items.append(('inst',inst))

def expr_vars(e, out):
    t=e[0]
    if t=='var': out.add(e[1])
    elif t in ('succ','pred'): expr_vars(e[1],out)
    elif t=='call':
        for a in e[2]: expr_vars(a,out)

def stmt_vars(s,out):
    t=s[0]
    if t=='assign': out.add(s[1]); expr_vars(s[2],out)
    elif t=='if':
        expr_vars(s[1],out)
        for z in s[2]: stmt_vars(z,out)
        for z in s[3]: stmt_vars(z,out)
    elif t=='while':
        expr_vars(s[1],out)
        for z in s[2]: stmt_vars(z,out)

def ordered_locals(fn):
    seen=set(fn['args']); order=[]
    def ev(e):
        t=e[0]
        if t=='var':
            v=e[1]
            if v not in seen: seen.add(v); order.append(v)
        elif t in ('succ','pred'): ev(e[1])
        elif t=='call':
            for a in e[2]: ev(a)
    def sv(s):
        t=s[0]
        if t=='assign':
            v=s[1]
            if v not in seen: seen.add(v); order.append(v)
            ev(s[2])
        elif t=='if':
            ev(s[1]); [sv(z) for z in s[2]]; [sv(z) for z in s[3]]
        elif t=='while': ev(s[1]); [sv(z) for z in s[2]]
    for s in fn['body']: sv(s)
    ev(fn['ret'])
    return order

def compile_program(funcs, entry):
    fidx={f['name']:i+1 for i,f in enumerate(funcs)}
    compiled=[]; frame_sizes={}
    maps={}
    for f in funcs:
        locs=ordered_locals(f)
        rho={a:i+1 for i,a in enumerate(f['args'])}
        nxt=len(f['args'])+1
        for v in locs:
            if v not in rho:
                rho[v]=nxt; nxt+=1
        maps[f['name']]=rho
        frame_sizes[f['name']]=max([0]+list(rho.values()))+1

    def compile_fn(f):
        rho=maps[f['name']]; k=len(f['args']); m=max([0]+list(rho.values())); p=m+1
        A=Asm()
        def emit_inc(box,target): A.emit([1,box,target])
        def emit_dec(box,t,e): A.emit([2,box,t,e])
        def clear_box(box, cont):
            q=A.fresh('clr'); A.label(q); A.emit([2,box,q,cont])
        def move_box(src,dst,cont):
            q=A.fresh('mv'); A.label(q)
            q2=A.fresh('mv2')
            A.emit([2,src,q2,cont]); A.label(q2); A.emit([1,dst,q])
        def copy_box(src,dst,scratch,cont):
            # dst,scratch initially 0; preserve src
            loop=A.fresh('cp'); got=A.fresh('cp_got'); restore=A.fresh('cp_rest'); restgot=A.fresh('cp_rg')
            A.label(loop); A.emit([2,src,got,restore])
            A.label(got); A.emit([1,dst,A.fresh('tmp')]); tmp=A.items[-1][1][2]; A.label(tmp); A.emit([1,scratch,loop])
            A.label(restore); A.emit([2,scratch,restgot,cont])
            A.label(restgot); A.emit([1,src,restore])
        def cexpr(e,base,cont):
            t=e[0]
            if t=='const':
                n=e[1]
                cur=None
                if n==0:
                    A.emit([1,base,A.fresh('c0')]) # hack + then - to jump while keep zero? no unconditional jump
                    l=A.items[-1][1][2]; A.label(l); A.emit([2,base,cont,cont])
                else:
                    # chain n increments
                    for j in range(n):
                        nxtl=cont if j==n-1 else A.fresh('c')
                        A.emit([1,base,nxtl])
                        if j<n-1: A.label(nxtl)
            elif t=='var':
                copy_box(rho[e[1]],base,base+1,cont)
            elif t in ('succ','pred'):
                mid=A.fresh(t); cexpr(e[1],base,mid); A.label(mid)
                if t=='succ': A.emit([1,base,cont])
                else: A.emit([2,base,cont,cont])
            elif t=='call':
                fname,args=e[1],e[2]
                def carg(i):
                    if i==len(args):
                        after=A.fresh('aftercall')
                        A.emit([3,fidx[fname],after,base])
                        A.label(after)
                        # clear argument boxes base+1..base+k sequentially
                        def clear_i(j):
                            if j>len(args):
                                # result already in base
                                # make unconditional via +,- on scratch base+len(args)+1
                                return cont
                            nxt=A.fresh('ca') if j<len(args) else cont
                            clear_box(base+j,nxt)
                            if j<len(args): A.label(nxt); clear_i(j+1)
                        if args: clear_i(1)
                        # no args -> need no op, but already at after label and fall-through impossible in graph; use jump macro
                        if not args:
                            A.emit([1,base+1,A.fresh('j')]); l=A.items[-1][1][2]; A.label(l); A.emit([2,base+1,cont,cont])
                        return
                    nxt=A.fresh('arg')
                    cexpr(args[i],base+1+i,nxt); A.label(nxt); carg(i+1)
                carg(0)
            else: raise ValueError(e)
        def cstmt(s,cont):
            t=s[0]
            if t=='assign':
                mid=A.fresh('as'); cexpr(s[2],p,mid); A.label(mid)
                clrdone=A.fresh('asclr'); clear_box(rho[s[1]],clrdone); A.label(clrdone)
                move_box(p,rho[s[1]],cont)
            elif t=='if':
                c_lab=A.fresh('ifc'); true=A.fresh('ift'); false=A.fresh('iff')
                cexpr(s[1],p,c_lab); A.label(c_lab)
                # branch nonzero and clear p before true
                cleartrue=A.fresh('ifclear')
                A.emit([2,p,cleartrue,false]); A.label(cleartrue)
                clear_box(p,true)
                A.label(true); cblock(s[2],cont)
                A.label(false); cblock(s[3],cont)
            elif t=='while':
                test=A.fresh('wtest'); c_lab=A.fresh('wc'); body=A.fresh('wbody'); done=A.fresh('wdone'); cleartrue=A.fresh('wclear')
                # jump to test using +,- scratch p
                A.emit([1,p,test]); A.label(test); A.emit([2,p,c_lab,c_lab]); A.label(c_lab)
                cexpr(s[1],p,cleartrue); A.label(cleartrue)
                # branch
                clearloop=A.fresh('wcloop')
                A.emit([2,p,clearloop,done]); A.label(clearloop)
                clear_box(p,body)
                A.label(body); cblock(s[2],test)
                A.label(done)
                # unconditional jump to cont
                A.emit([1,p,A.fresh('wdj')]); l=A.items[-1][1][2]; A.label(l); A.emit([2,p,cont,cont])
            else: raise ValueError(s)
        def cblock(stmts,cont):
            if not stmts:
                # unconditional jump via p
                A.emit([1,p,A.fresh('bj')]); l=A.items[-1][1][2]; A.label(l); A.emit([2,p,cont,cont]); return
            # compile in sequence with labels
            next_labels=[A.fresh('s') for _ in range(len(stmts)-1)]
            for i,s in enumerate(stmts):
                cc=cont if i==len(stmts)-1 else next_labels[i]
                cstmt(s,cc)
                if i<len(stmts)-1: A.label(next_labels[i])
        retlab=A.fresh('ret')
        cblock(f['body'],retlab); A.label(retlab)
        retready=A.fresh('retready'); cexpr(f['ret'],p,retready); A.label(retready)
        clr0=A.fresh('clr0'); clear_box(0,clr0); A.label(clr0)
        moved=A.fresh('retm'); move_box(p,0,moved); A.label(moved)
        # clear locals k+1..m
        cur=moved
        # we're already at moved label. emit sequential clear
        for b in range(k+1,m+1):
            nxt=A.fresh('locclr')
            clear_box(b,nxt); A.label(nxt)
        A.emit([4])
        # resolve labels local
        labels={}; pc=0
        for it in A.items:
            if it[0]=='label': labels[it[1]]=pc
            else: pc+=1
        code=[]
        for it in A.items:
            if it[0]=='inst':
                inst=[]
                for v in it[1]: inst.append(labels[v] if isinstance(v,str) else v)
                code.append(inst)
        return code

    codes=[]
    for f in funcs:
        codes.append(compile_fn(f))
    main=[[3,fidx[entry],1,0],[0]]
    funs=[0]
    off=len(main)
    for code in codes:
        funs.append(off); off+=len(code)
    return main+sum(codes,[]), funs, fidx, frame_sizes, maps, [len(c) for c in codes]



このコンパイラでは、関数呼び出しを呼び出し箇所へ展開しません。各関数を 1 回だけコンパイルし、呼び出し式は `[3,a,next,m]` に落とします。この点が、通常のアバカスマシンへ展開する方式との大きな違いです。

## 3. `UNIVS` を構成する WHILE プログラム

`UNIVS` だけでなく、それが使う `add`, `mul`, `div`, `mod`, `cons`, `head`, `tail`, `listentry`, `listreplace` も同じ WHILE AST で記述します。したがって、これらは呼び出しごとに複製されず、それぞれ一つの関数としてコンパイルされます。

In [2]:
# Source program definitions
add=F('add',['x','y'],[
    Assign('r',C(0)),
    While(V('x'),[Assign('x',Pred(V('x'))),Assign('r',Succ(V('r')))]),
    While(V('y'),[Assign('y',Pred(V('y'))),Assign('r',Succ(V('r')))]),
],V('r'))
sub=F('sub',['x','y'],[
    While(V('y'),[
        If(V('x'),[Assign('x',Pred(V('x'))),Assign('y',Pred(V('y')))], [Assign('y',C(0))])
    ])
],V('x'))
le=F('le',['x','y'],[
    Assign('r',C(1)),
    While(V('x'),[
        If(V('y'),[Assign('x',Pred(V('x'))),Assign('y',Pred(V('y')))], [Assign('r',C(0)),Assign('x',C(0))])
    ])
],V('r'))
eq=F('eq',['x','y'],[
    Assign('a',Call('le',V('x'),V('y'))),
    Assign('b',Call('le',V('y'),V('x'))),
    Assign('r',Call('mul',V('a'),V('b'))),
],V('r'))
mul=F('mul',['x','y'],[
    Assign('r',C(0)),
    While(V('y'),[
        Assign('y',Pred(V('y'))),
        Assign('r',Call('add',V('r'),V('x'))),
    ])
],V('r'))
div=F('div',['x','y'],[
    Assign('q',C(0)),
    Assign('c',Call('le',V('y'),V('x'))),
    While(V('c'),[
        Assign('x',Call('sub',V('x'),V('y'))),
        Assign('q',Succ(V('q'))),
        Assign('c',Call('le',V('y'),V('x'))),
    ])
],V('q'))
mod=F('mod',['x','y'],[
    Assign('c',Call('le',V('y'),V('x'))),
    While(V('c'),[
        Assign('x',Call('sub',V('x'),V('y'))),
        Assign('c',Call('le',V('y'),V('x'))),
    ])
],V('x'))
cons=F('cons',['a','x'],[
    Assign('n',Succ(V('a'))), Assign('p',C(1)),
    Assign('t',Call('mul',C(2),V('p'))), Assign('c',Call('le',V('t'),V('n'))),
    While(V('c'),[
        Assign('p',Call('mul',C(2),V('p'))),
        Assign('t',Call('mul',C(2),V('p'))),
        Assign('c',Call('le',V('t'),V('n'))),
    ]),
    Assign('r',Call('sub',V('n'),V('p'))),
    Assign('u',Call('mul',C(2),V('r'))), Assign('u',Succ(V('u'))),
    Assign('v',Call('mul',C(2),V('p'))), Assign('v',Call('mul',V('v'),V('x'))),
    Assign('u',Call('add',V('u'),V('v'))), Assign('ans',Call('mul',V('p'),V('u'))),
],V('ans'))
head=F('head',['x'],[
    Assign('ans',C(0)),
    If(V('x'),[
        Assign('p',C(1)),
        Assign('m',Call('mod',V('x'),C(2))), Assign('even',Call('eq',V('m'),C(0))),
        While(V('even'),[
            Assign('x',Call('div',V('x'),C(2))), Assign('p',Call('mul',V('p'),C(2))),
            Assign('m',Call('mod',V('x'),C(2))), Assign('even',Call('eq',V('m'),C(0))),
        ]),
        Assign('y',Call('div',Pred(V('x')),C(2))), Assign('r',Call('mod',V('y'),V('p'))),
        Assign('ans',Pred(Call('add',V('p'),V('r')))),
    ],[])
],V('ans'))
tail=F('tail',['x'],[
    Assign('ans',C(0)),
    If(V('x'),[
        Assign('p',C(1)), Assign('m',Call('mod',V('x'),C(2))), Assign('even',Call('eq',V('m'),C(0))),
        While(V('even'),[
            Assign('x',Call('div',V('x'),C(2))), Assign('p',Call('mul',V('p'),C(2))),
            Assign('m',Call('mod',V('x'),C(2))), Assign('even',Call('eq',V('m'),C(0))),
        ]),
        Assign('y',Call('div',Pred(V('x')),C(2))), Assign('ans',Call('div',V('y'),V('p'))),
    ],[])
],V('ans'))
listentry=F('listentry',['x','k'],[
    While(V('k'),[Assign('x',Call('tail',V('x'))),Assign('k',Pred(V('k')))]),
    Assign('r',Call('head',V('x'))),
],V('r'))
listreplace=F('listreplace',['x','k','a'],[
    Assign('kk',V('k')), Assign('u',C(0)),
    While(V('k'),[
        Assign('h',Call('head',V('x'))), Assign('u',Call('cons',V('h'),V('u'))),
        Assign('x',Call('tail',V('x'))), Assign('k',Pred(V('k'))),
    ]),
    Assign('t',Call('tail',V('x'))), Assign('x',Call('cons',V('a'),V('t'))),
    While(V('kk'),[
        Assign('h',Call('head',V('u'))), Assign('x',Call('cons',V('h'),V('x'))),
        Assign('u',Call('tail',V('u'))), Assign('kk',Pred(V('kk'))),
    ]),
],V('x'))
univs=F('UNIVS',['p','funs','d'],[
    Assign('pc',C(0)),Assign('fp',C(0)),Assign('fun',C(0)),Assign('s',C(0)),
    Assign('inst',Call('listentry',V('p'),V('pc'))),Assign('op',Call('listentry',V('inst'),C(0))),
    While(V('op'),[
        Assign('box',Call('listentry',V('inst'),C(1))),Assign('next',Call('listentry',V('inst'),C(2))),Assign('enext',Call('listentry',V('inst'),C(3))),
        If(Call('eq',V('op'),C(1)),[
            Assign('i',Call('add',V('box'),V('fp'))),Assign('v',Call('listentry',V('d'),V('i'))),Assign('v',Succ(V('v'))),
            Assign('d',Call('listreplace',V('d'),V('i'),V('v'))),Assign('base',Call('listentry',V('funs'),V('fun'))),Assign('pc',Call('add',V('base'),V('next'))),
        ],[]),
        If(Call('eq',V('op'),C(2)),[
            Assign('i',Call('add',V('box'),V('fp'))),Assign('v',Call('listentry',V('d'),V('i'))),
            If(V('v'),[
                Assign('v',Pred(V('v'))),Assign('d',Call('listreplace',V('d'),V('i'),V('v'))),Assign('base',Call('listentry',V('funs'),V('fun'))),Assign('pc',Call('add',V('base'),V('next'))),
            ],[
                Assign('base',Call('listentry',V('funs'),V('fun'))),Assign('pc',Call('add',V('base'),V('enext'))),
            ])
        ],[]),
        If(Call('eq',V('op'),C(3)),[
            Assign('t',Call('cons',V('fp'),C(0))),Assign('t',Call('cons',V('next'),V('t'))),Assign('frame',Call('cons',V('fun'),V('t'))),Assign('s',Call('cons',V('frame'),V('s'))),
            Assign('fp',Call('add',V('fp'),V('enext'))),Assign('fun',V('box')),Assign('pc',Call('listentry',V('funs'),V('fun'))),
        ],[]),
        If(Call('eq',V('op'),C(4)),[
            Assign('frame',Call('head',V('s'))),Assign('fun',Call('listentry',V('frame'),C(0))),Assign('next',Call('listentry',V('frame'),C(1))),Assign('fp',Call('listentry',V('frame'),C(2))),Assign('s',Call('tail',V('s'))),
            Assign('base',Call('listentry',V('funs'),V('fun'))),Assign('pc',Call('add',V('base'),V('next'))),
        ],[]),
        Assign('inst',Call('listentry',V('p'),V('pc'))),Assign('op',Call('listentry',V('inst'),C(0))),
    ]),
    Assign('r',Call('listentry',V('d'),C(0))),
],V('r'))
funcs=[add,sub,mul,le,eq,div,mod,cons,head,tail,listentry,listreplace,univs]



## 4. `listcode` とスタック式アバカスマシンの実行器

In [3]:
# list coding python helpers
def cons_py(a,x):
    n=a+1; p=1
    while 2*p<=n: p*=2
    r=n-p
    return p*(2*r+1+2*p*x)
def listcode_py(xs):
    x=0
    for a in reversed(xs): x=cons_py(a,x)
    return x

def run_machine(program,funs,args,max_steps=20_000_000):
    d=[0]+list(args); pc=fp=fun=0; stack=[]; steps=0
    def get(i): return d[i] if i<len(d) else 0
    def put(i,v):
        while len(d)<=i: d.append(0)
        d[i]=v
    while program[pc][0]>0:
        steps+=1
        if steps>max_steps: raise RuntimeError(('step limit',pc,fun,fp,d[:20]))
        ins=program[pc]; op=ins[0]
        box=ins[1] if len(ins)>1 else 0; nx=ins[2] if len(ins)>2 else 0; ex=ins[3] if len(ins)>3 else 0
        if op==1: put(fp+box,get(fp+box)+1); pc=funs[fun]+nx
        elif op==2:
            if get(fp+box)>0: put(fp+box,get(fp+box)-1); pc=funs[fun]+nx
            else: pc=funs[fun]+ex
        elif op==3: stack.append((fun,nx,fp)); fp+=ex; fun=box; pc=funs[fun]
        elif op==4: fun,nx,fp=stack.pop(); pc=funs[fun]+nx
    return d[0],steps,d



## 5. `UNIVS` を実際にコンパイルする

In [4]:
prog, funs, fidx, frame_sizes, var_maps, function_lengths = compile_program(funcs, 'UNIVS')

names = ['main'] + [f['name'] for f in funcs]
lengths = [2] + function_lengths

print('関数数（main を含む） =', len(names))
print('総命令数 =', len(prog))
print('funs =', funs)
print()
for name, n in zip(names, lengths):
    print(f'{name:12s} {n:5d} instructions')


関数数（main を含む） = 14
総命令数 = 1837
funs = [0, 2, 75, 125, 176, 236, 296, 380, 449, 660, 829, 980, 1030, 1194]

main             2 instructions
add             73 instructions
sub             50 instructions
mul             51 instructions
le              60 instructions
eq              60 instructions
div             84 instructions
mod             69 instructions
cons           211 instructions
head           169 instructions
tail           151 instructions
listentry       50 instructions
listreplace    164 instructions
UNIVS          643 instructions


ここで重要なのは、`UNIVS` 本体だけでなく、符号化されたリストを操作するライブラリまで含めても数千命令規模に収まることです。関数呼び出しを展開せず CALL/RET にした効果が大きく出ます。

## 6. コンパイル済み `UNIVS` のコードの大きさ

In [5]:
import math

p_code = listcode_py([listcode_py(inst) for inst in prog])
funs_code = listcode_py(funs)

p_bits = p_code.bit_length()
p_digits = int(p_bits * math.log10(2)) + 1 if p_code else 1

print('命令数                 =', len(prog))
print('プログラムコード bit 数 =', p_bits)
print('10進桁数（推定）        =', p_digits)
print('funs のコード bit 数    =', funs_code.bit_length())


命令数                 = 1837
プログラムコード bit 数 = 100043
10進桁数（推定）        = 30116
funs のコード bit 数    = 208


10進数として全部表示すると長いので、ここでは桁数だけを表示します。プログラムコード自体は巨大な「値」ですが、記述長としては数万桁程度です。問題はサイズよりも、これを unary なアバカスマシン命令だけで処理するときの実行時間です。

## 7. コンパイル済み関数を少し動かして確認する

In [6]:
def run_compiled(name, args, max_steps=5_000_000):
    p = [inst[:] for inst in prog]
    p[0] = [3, fidx[name], 1, 0]
    return run_machine(p, funs, args, max_steps=max_steps)

for name, args in [
    ('add',[2,3]),
    ('sub',[5,2]),
    ('eq',[3,3]),
    ('div',[7,2]),
    ('mod',[7,2]),
    ('head',[2]),
    ('tail',[2]),
]:
    value, steps, _ = run_compiled(name, args)
    print(f'{name}{tuple(args)} = {value}   ({steps} machine steps)')


add(2, 3) = 5   (341 machine steps)
sub(5, 2) = 3   (228 machine steps)
eq(3, 3) = 1   (711 machine steps)
div(7, 2) = 3   (1903 machine steps)
mod(7, 2) = 1   (1836 machine steps)
head(2,) = 1   (2222 machine steps)
tail(2,) = 0   (2078 machine steps)


## 8. `UNIVS in UNIVS` — 最小の自己解釈実験

ここが中心です。

内側のプログラムとして、まず最小の

```text
0: [0]
```

だけを与えます。これは何もしないで終了するスタック式アバカスマシンプログラムです。

外側では、先ほどコンパイルした **スタック式アバカスマシン版 `UNIVS`** を実行します。その入力として、内側の `p`, `funs`, `d` をすべて `listcode` した自然数を渡します。

したがって、実際に

```text
スタック式アバカスマシン
  └─ コンパイル済み UNIVS
       └─ 内側のスタック式アバカスマシンプログラム [0]
```

という二重構造になります。


In [7]:
inner_program = [[0]]
inner_funs = [0]
inner_data = [0]

inner_p_code = listcode_py([listcode_py(inst) for inst in inner_program])
inner_funs_code = listcode_py(inner_funs)
inner_d_code = listcode_py(inner_data)

value, steps, _ = run_compiled(
    'UNIVS',
    [inner_p_code, inner_funs_code, inner_d_code],
    max_steps=1_000_000,
)

print('inner p code    =', inner_p_code)
print('inner funs code =', inner_funs_code)
print('inner data code =', inner_d_code)
print('result          =', value)
print('outer machine steps =', steps)


inner p code    = 2
inner funs code = 1
inner data code = 1
result          = 0
outer machine steps = 3737


この最小例は実際に完走します。つまり、**コンパイルされた `UNIVS` が、別のスタック式アバカスマシンプログラムを解釈する**ところまでは現実に動きます。

ただし、内側のプログラムを少し大きくすると急に重くなります。たとえば

```text
0: [1,0,1]
1: [0]
```

のように、箱 0 に 1 を加えるだけでも、内側プログラムの `listcode` が大きくなり、その自然数を `head`, `tail`, `div`, `mod` などを通して unary に処理するため非常に遅くなります。


## 9. 重い実験（デフォルトでは実行しない）

In [8]:
# 次を試すと、非常に多くのステップが必要になります。
# inner_program = [[1,0,1],[0]]
# inner_p_code = listcode_py([listcode_py(inst) for inst in inner_program])
# value, steps, _ = run_compiled(
#     'UNIVS',
#     [inner_p_code, listcode_py([0]), listcode_py([0])],
#     max_steps=50_000_000,
# )
# print(value, steps)


## 10. `Lisp in Lisp` との対応と、次の self-hosting 段階

対応を並べると、かなり似た構図になります。

| Lisp 的な見方 | この notebook |
|---|---|
| Lisp の式 | WHILE の構文木 |
| eval | `UNIVS` |
| Lisp で eval を記述 | WHILE で `UNIVS` を記述 |
| Lisp を Lisp で動かす | コンパイル済み `UNIVS` で別の機械語プログラムを解釈 |
| self-hosting compiler | WHILE 自身で `COMP` を記述して `COMP` をコンパイル |

この notebook では、最後の **コンパイラ自身を WHILE で書いて自己コンパイルする部分**はまだ Python 実装で代用しています。しかし、`COMP` が行っていることは、構文木の有限列操作、ラベルの管理、命令列の連結、自然数の計算だけです。したがって WHILE 言語へ移せます。

次の段階は

```text
WHILE で書いた COMP
      ↓ COMP で自己コンパイル
スタック式アバカスマシン版 COMP
```

を作ることです。そこまで行くと、かなり文字通りの **「手続型言語版 Lisp in Lisp」** になります。
